In [1]:
# pip install openai chromadb python-dotenv requests pandas edgartools

In [2]:
import os
import time
import requests
import chromadb
from dotenv import load_dotenv
from openai import OpenAI
from edgar import Company, set_identity

load_dotenv()
client_ai = OpenAI()
set_identity("Isaiah Lacet isaiahlacet@gmail.com")

EMBEDDING_MODEL = "text-embedding-3-small"

chroma_client = chromadb.PersistentClient(path="vector_store")
collection = chroma_client.get_or_create_collection(name="filings")

In [3]:
HEADERS = {"User-Agent": "Isaiah Lacet isaiahlacet@gmail.com"}
_ticker_to_cik = None

def _load_ticker_map():
    global _ticker_to_cik
    if _ticker_to_cik is None:
        data = requests.get("https://www.sec.gov/files/company_tickers.json", headers=HEADERS).json()
        _ticker_to_cik = {v["ticker"]: str(v["cik_str"]).zfill(10) for v in data.values()}
    return _ticker_to_cik

def is_valid_ticker(ticker):
    return ticker.upper() in _load_ticker_map()

In [4]:
def get_latest_10k(ticker):
    ticker = ticker.upper()
    filings = [f for f in Company(ticker).get_filings(form="10-K") if f.form == "10-K"]
    if not filings:
        raise ValueError(f"No original (non-amended) 10-K filings found for '{ticker}'.")
    return filings[0]

In [5]:
def extract_sections(tenk, ticker):
    chunks = []
    for item in tenk.items:
        try:
            text = tenk[item]
        except Exception:
            continue
        if text and len(text.strip()) > 0:
            chunks.append({"ticker": ticker, "section": item, "text": text.strip()})
    return chunks

In [6]:
def split_long_text(text, max_chars=6000):
    if len(text) <= max_chars:
        return [text]
    parts = []
    paragraphs = text.split("\n")
    current = ""
    for para in paragraphs:
        if len(current) + len(para) + 1 > max_chars:
            if current:
                parts.append(current.strip())
            current = para
        else:
            current += "\n" + para if current else para
    if current:
        parts.append(current.strip())
    return parts

In [7]:
def looks_like_a_10k(chunks):
    for c in chunks:
        if "1A" in c["section"] and len(c["text"]) > 300:
            return True
    return False

In [8]:
def get_embedding(text, model=EMBEDDING_MODEL):
    resp = client_ai.embeddings.create(input=[text], model=model)
    return resp.data[0].embedding

def get_embeddings(texts, model=EMBEDDING_MODEL, batch_size=100):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        resp = client_ai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([e.embedding for e in resp.data])
    return all_embeddings

In [9]:
def is_ticker_indexed(ticker, accession_no=None):
    ticker = ticker.upper()
    existing = collection.get(where={"ticker": ticker}, limit=1)
    if len(existing["ids"]) == 0:
        return False
    if accession_no is None:
        return True
    indexed_accession = existing["metadatas"][0].get("accession_no")
    return indexed_accession == accession_no

def ensure_ticker_indexed(ticker):
    ticker = ticker.upper()

    if not is_valid_ticker(ticker):
        raise ValueError(f"'{ticker}' is not a recognized ticker in the SEC's ticker list.")

    filing = get_latest_10k(ticker)
    accession_no = filing.accession_no

    if is_ticker_indexed(ticker, accession_no=accession_no):
        return  # already indexed and it's the current filing

    existing = collection.get(where={"ticker": ticker})
    if existing["ids"]:
        print(f"Newer 10-K found for {ticker} — re-indexing...")
        collection.delete(ids=existing["ids"])

    print(f"Indexing latest 10-K for {ticker}...")
    tenk = filing.obj()

    raw_chunks = extract_sections(tenk, ticker)
    if not raw_chunks:
        raise ValueError(f"Parsed 0 sections from {ticker}'s 10-K.")
    if not looks_like_a_10k(raw_chunks):
        raise ValueError(f"Parsed sections for {ticker} don't look like a standard 10-K — manual review needed.")

    split_chunks = []
    for c in raw_chunks:
        pieces = split_long_text(c["text"])
        for i, piece in enumerate(pieces):
            split_chunks.append({
                "ticker": c["ticker"],
                "section": c["section"] if len(pieces) == 1 else f"{c['section']} (part {i+1})",
                "text": piece
            })

    texts = [c["text"] for c in split_chunks]
    embeddings = get_embeddings(texts)
    ids = [f"{ticker}_{i}" for i in range(len(split_chunks))]
    metadatas = [{"ticker": c["ticker"], "section": c["section"], "accession_no": accession_no} for c in split_chunks]

    collection.add(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)
    print(f"Indexed {len(split_chunks)} chunks for {ticker} (accession {accession_no}).")

In [10]:
def extract_tickers(question, model="gpt-4o-mini"):
    prompt = f"""Extract the stock ticker symbol(s) for any publicly traded company mentioned in this question.
Return ONLY the ticker symbols, comma-separated, uppercase, nothing else.
If no company is mentioned, return NONE.

Question: {question}"""
    resp = client_ai.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    raw = resp.choices[0].message.content.strip()
    if raw == "NONE":
        return []
    return [t.strip().upper() for t in raw.split(",")]

def validate_tickers(tickers):
    return [t for t in tickers if is_valid_ticker(t)]

In [11]:
def retrieve(query, query_embedding=None, n_per_ticker=20, tickers=None):
    if not tickers:
        raise ValueError("retrieve() requires at least one ticker in `tickers`.")
    if query_embedding is None:
        query_embedding = get_embedding(query)

    all_results = []
    for ticker in tickers:
        ensure_ticker_indexed(ticker)
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=n_per_ticker,
            where={"ticker": ticker.upper()}
        )
        for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
            all_results.append({"text": doc, "ticker": meta["ticker"], "section": meta["section"]})
    return all_results

def ask(question, query_embedding=None, relevant_tickers=None, n_per_ticker=20, model="gpt-4o-mini"):
    if not relevant_tickers:
        relevant_tickers = validate_tickers(extract_tickers(question))
        if not relevant_tickers:
            raise ValueError("Couldn't identify a valid company/ticker in the question. Try passing relevant_tickers explicitly.")
        print(f"Auto-detected ticker(s): {relevant_tickers}")

    tickers = tuple(t.upper() for t in relevant_tickers)
    chunks = retrieve(question, query_embedding=query_embedding, n_per_ticker=n_per_ticker, tickers=tickers)
    context = "\n\n".join(f"[{c['ticker']} - {c['section']}]\n{c['text']}" for c in chunks)

    prompt = f"""Answer the question using only the context below, which is pulled from SEC 10-K filings.
If the context doesn't contain the answer, say so instead of guessing. Cite the ticker for any claim you make.

Context:
{context}

Question: {question}

Answer:"""

    for attempt in range(5):
        try:
            resp = client_ai.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )
            return resp.choices[0].message.content, chunks
        except Exception as e:
            if "rate_limit" in str(e).lower() or "429" in str(e):
                wait = 2 ** attempt * 5
                print(f"Rate limited, waiting {wait}s (attempt {attempt+1}/5)...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Failed after 5 retries due to rate limiting")

In [12]:
question = "What does NVIDIA say about its reliance on third-party suppliers?"
answer, sources = ask(question)

print("ANSWER:\n", answer)
print("\nSOURCES USED:")
for c in sources:
    print(f"- {c['ticker']}: {c['section']}")

Auto-detected ticker(s): ['NVDA']
ANSWER:
 NVIDIA states that it depends on foundries to manufacture its semiconductor wafers and does not assemble, test, or package its products, but instead contracts with independent subcontractors for these processes. This reliance on third-party suppliers reduces NVIDIA's control over product quantity and quality, manufacturing yields, and delivery schedules, which could harm its business. The company faces risks that could adversely affect its ability to meet customer demand and scale its supply chain, negatively impacting longer-term demand for its products and services, as well as its business operations, gross margin, revenue, and financial results. These risks include lack of guaranteed supply of components, potential higher prices due to demand estimation errors, and disruptions in manufacturing processes caused by various factors.

SOURCES USED:
- NVDA: Item 7 (part 1)
- NVDA: Item 1 (part 2)
- NVDA: Item 1 (part 3)
- NVDA: Item 1A (part 3)


In [13]:
question = "What risks does Amazon disclose related to its logistics and delivery network?"
answer, sources = ask(question)

print("ANSWER:\n", answer)
print("\nSOURCES USED:")
for c in sources:
    print(f"- {c['ticker']}: {c['section']}")

Auto-detected ticker(s): ['AMZN']
ANSWER:
 Amazon discloses several risks related to its logistics and delivery network, including:

1. **Inventory Management Risks**: Amazon faces significant inventory risks that may adversely affect its operating results due to seasonality, new product launches, rapid changes in product cycles and pricing, defective merchandise, changes in customer demand, and spoilage. The company endeavors to accurately predict trends to avoid overstocking or understocking products.

2. **Fulfillment Network Optimization**: Failures to adequately predict customer demand and optimize the fulfillment network can result in excess or insufficient capacity, service interruptions, increased costs, and impairment charges.

3. **Staffing Challenges**: The company may struggle to adequately staff its fulfillment network and customer service centers, particularly during peak periods, which can negatively impact operational efficiency.

4. **Dependence on Shipping Companies**

In [14]:
question = "What does Costco disclose about risks related to membership fee renewal rates?"
answer, sources = ask(question, relevant_tickers=["COST"])

print("ANSWER:\n", answer)
print("\nSOURCES USED:")
for c in sources:
    print(f"- {c['ticker']}: {c['section']}")

ANSWER:
 Costco discloses that membership loyalty and growth are essential to its business, and the extent to which it achieves growth in its membership base, increases the penetration of Executive membership, and sustains high renewal rates materially influences its profitability. Damage to its brands or reputation may negatively impact comparable sales, diminish member trust, and reduce renewal rates, which could adversely affect net sales and membership fee revenue, ultimately impacting the company's results of operations.

SOURCES USED:
- COST: Item 1 (part 3)
- COST: Item 8 (part 8)
- COST: Item 16
- COST: Item 8 (part 4)
- COST: Item 8 (part 14)
- COST: Item 5
- COST: Item 4
- COST: Item 11
- COST: Item 1 (part 2)
- COST: Item 1 (part 1)
- COST: Item 13
- COST: Item 8 (part 13)
- COST: Item 12
- COST: Item 10
- COST: Item 8 (part 2)
- COST: Item 1A (part 3)
- COST: Item 8 (part 1)
- COST: Item 1A (part 8)
- COST: Item 1A (part 1)
- COST: Item 15


In [15]:
question = "What risks does Disney disclose related to its streaming and direct-to-consumer business?"
answer, sources = ask(question, relevant_tickers=["DIS"])

print("ANSWER:\n", answer)
print("\nSOURCES USED:")
for c in sources:
    print(f"- {c['ticker']}: {c['section']}")

ANSWER:
 Disney discloses several risks related to its streaming and direct-to-consumer (DTC) business, including:

1. **Competition**: Disney faces substantial competition from alternative providers of products and services, including other DTC and linear offerings, which can impact viewer and subscriber numbers.

2. **Content Curation and Investment Decisions**: The success of DTC services is heavily dependent on the effectiveness of content curation and investment decisions, which may not always attract and retain subscribers as expected.

3. **Subscriber Growth and Churn**: There is a risk of flat subscriber growth or net losses of subscribers, which can be exacerbated by economic conditions that affect consumer willingness to pay for multiple DTC services.

4. **Cost Management**: The company may struggle to manage costs effectively, especially as content costs increase and competition for creative talent and programming rights intensifies.

5. **Regulatory Changes**: Government r

In [16]:
question = "What does Tesla say about its battery supply chain?"
answer, sources = ask(question, n_per_ticker=30)

print("ANSWER:\n", answer)
print("\nSOURCES USED:")
for c in sources:
    print(f"- {c['ticker']}: {c['section']}")

Auto-detected ticker(s): ['TSLA']
ANSWER:
 Tesla states that it is dependent on the continued supply of lithium-ion battery cells for its vehicles and energy storage products, and it requires substantially more cells to grow its business according to its plans. Currently, Tesla relies on suppliers such as Panasonic and Contemporary Amperex Technology Co. Limited (CATL) for these cells. The company has fully qualified only a limited number of such suppliers and has limited flexibility in changing suppliers. Any disruption in the supply of battery cells from these suppliers could limit production of Tesla's vehicles and energy storage products. In the long term, Tesla intends to supplement cells from its suppliers with cells manufactured by itself, which it believes will be more efficient and cost-effective. However, the development and manufacture of these battery cells require significant investments, and there is no assurance that Tesla will achieve its targets in the planned timefram

In [17]:
question = "What does Nike say about the risk of counterfeit products harming its brand?"
answer, sources = ask(question, n_per_ticker=30)

print("ANSWER:\n", answer)
print("\nSOURCES USED:")
for c in sources:
    print(f"- {c['ticker']}: {c['section']}")

Auto-detected ticker(s): ['NKE']
ANSWER:
 Nike states that it periodically discovers counterfeit reproductions of its products or products that otherwise infringe its intellectual property rights. If Nike is unsuccessful in enforcing its intellectual property rights, the continued sales of these counterfeit products could adversely affect its sales and brand, potentially resulting in a shift of consumer preference away from Nike's products. This indicates that the risk of counterfeit products poses a significant threat to Nike's brand image and market position. 

(NKE - Item 1A (part 15))

SOURCES USED:
- NKE: Item 1A (part 2)
- NKE: Item 1 (part 4)
- NKE: Item 1A (part 1)
- NKE: Item 1A (part 12)
- NKE: Item 1C
- NKE: Item 1A (part 3)
- NKE: Item 1 (part 3)
- NKE: Item 1A (part 17)
- NKE: Item 1A (part 10)
- NKE: Item 1 (part 1)
- NKE: Item 1A (part 6)
- NKE: Item 1A (part 9)
- NKE: Item 7 (part 1)
- NKE: Item 1A (part 8)
- NKE: Item 1A (part 7)
- NKE: Item 8 (part 20)
- NKE: Item 1 (